# Blog 9 — Failure Recovery in Data Engineering

## Complete Isolated Databricks Notebook

This notebook demonstrates how a production-style data pipeline can:

- detect failures
- classify failures
- track batch state
- retry transient failures
- replay failed batches
- use idempotent writes
- quarantine bad records
- audit recovery events
- validate final correctness

### Dedicated environment

```text
workspace.blog9_failure_recovery
├── orders
├── pipeline_control
├── quarantine
└── recovery_audit
```

No DBFS root, `/tmp`, or previous-blog tables are required.

### Core principle

> A reliable pipeline is not one that never fails. It is one that can fail, recover safely, and still reach the correct final state.

# 1. Failure Recovery — The Problem

Consider:

```text
Batch 101
   ↓
Read
   ↓
Transform
   ↓
Write
   ↓
💥 Failure
```

The important question is not simply:

> "Should we rerun?"

It is:

> **"What state is the system actually in?"**

We need to determine:

- Did the write happen?
- Was the Delta transaction committed?
- Was pipeline state updated?
- Were records duplicated?
- Is the failure transient or deterministic?
- Can the batch be safely replayed?

# 2. Three Failure Categories

| Failure type | Example | Typical action |
|---|---|---|
| Transient | temporary network/cluster issue | Retry |
| Deterministic | schema mismatch, code bug, permission error | Stop → Fix → Replay |
| Data-level | invalid individual records | Quarantine invalid records |

The important engineering principle is:

> **Retry transient failures; don't blindly retry deterministic failures.**

# 3. Delta Transactions vs Pipeline State

Two different states exist:

```text
Delta transaction state
        ≠
Pipeline orchestration state
```

A Delta write may succeed while the application crashes before updating the control table.

Example:

```text
Delta MERGE succeeds
       ↓
Application crashes
       ↓
Control table still says FAILED
```

This is one of the reasons idempotent replay is essential.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid
import json

CATALOG = "workspace"
SCHEMA = "blog9_failure_recovery"

ORDERS_TABLE = f"{CATALOG}.{SCHEMA}.orders"
CONTROL_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_control"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.quarantine"
AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.recovery_audit"

print("Catalog:", CATALOG)
print("Schema :", SCHEMA)
print("Spark  :", spark.version)

Catalog: workspace
Schema : blog9_failure_recovery
Spark  : 4.1.0


# 4. Create the Isolated Environment

The schema is recreated so the notebook can be rerun from the beginning.

In [0]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")

spark.sql(f'''
CREATE SCHEMA {CATALOG}.{SCHEMA}
COMMENT 'Isolated Blog 9 - Failure Recovery'
''')

print(f"Created {CATALOG}.{SCHEMA}")

Created workspace.blog9_failure_recovery


In [0]:
spark.sql(f"DESCRIBE SCHEMA EXTENDED {CATALOG}.{SCHEMA}").show(truncate=False)

+-------------------------+---------------------------------------------------------+
|database_description_item|database_description_value                               |
+-------------------------+---------------------------------------------------------+
|Catalog Name             |workspace                                                |
|Namespace Name           |blog9_failure_recovery                                   |
|Comment                  |Isolated Blog 9 - Failure Recovery                       |
|Location                 |                                                         |
|Owner                    |bharath2704.a@gmail.com                                  |
|Properties               |                                                         |
|Predictive Optimization  |ENABLE (inherited from METASTORE metastore_aws_us_east_2)|
+-------------------------+---------------------------------------------------------+



# 5. Create the Target Delta Table

The target uses a deterministic business key:

```text
order_id
```

This key will allow us to demonstrate idempotent replay.

In [0]:
orders_schema = StructType([
    StructField("order_id", LongType(), False),
    StructField("batch_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("amount", DoubleType(), False),
    StructField("status", StringType(), False)
])

empty_orders = spark.createDataFrame([], orders_schema)

empty_orders.write     .format("delta")     .mode("overwrite")     .saveAsTable(ORDERS_TABLE)

print("Created:", ORDERS_TABLE)

Created: workspace.blog9_failure_recovery.orders


# 6. Create the Pipeline Control Table

The control table records the operational state of each batch.

```text
PENDING
RUNNING
SUCCESS
FAILED
```

This is orchestration state, not Delta transaction history.

In [0]:
control_schema = StructType([
    StructField("batch_id", IntegerType(), False),
    StructField("status", StringType(), False),
    StructField("attempt", IntegerType(), False),
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("records_processed", LongType(), True),
    StructField("error_type", StringType(), True),
    StructField("error_message", StringType(), True)
])

spark.createDataFrame([], control_schema)     .write.format("delta")     .mode("overwrite")     .saveAsTable(CONTROL_TABLE)

print("Created:", CONTROL_TABLE)

Created: workspace.blog9_failure_recovery.pipeline_control


# 7. Create Quarantine and Recovery Audit Tables

In [0]:
quarantine_schema = StructType([
    StructField("batch_id", IntegerType(), False),
    StructField("order_id", LongType(), True),
    StructField("reason", StringType(), False),
    StructField("error_type", StringType(), False),
    StructField("created_at", TimestampType(), False)
])

spark.createDataFrame([], quarantine_schema)     .write.format("delta")     .mode("overwrite")     .saveAsTable(QUARANTINE_TABLE)

audit_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("batch_id", IntegerType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("event_type", StringType(), False),
    StructField("details", StringType(), True)
])

spark.createDataFrame([], audit_schema)     .write.format("delta")     .mode("overwrite")     .saveAsTable(AUDIT_TABLE)

print("Created quarantine and audit tables.")

Created quarantine and audit tables.


# 8. Generate Source Batches

We create five deterministic batches.

Each batch contains 100 orders.

```text
Batch 1 → 100
Batch 2 → 100
Batch 3 → 100
Batch 4 → 100
Batch 5 → 100
```

The deterministic `order_id` is what makes replay safe.

In [0]:
def create_batch(batch_id, rows_per_batch=100):
    rows = [
        (
            batch_id * 100000 + i,
            batch_id,
            (i % 50) + 1,
            float((i % 500) + 25),
            "COMPLETED"
        )
        for i in range(1, rows_per_batch + 1)
    ]
    return spark.createDataFrame(rows, orders_schema)

batches = {
    batch_id: create_batch(batch_id)
    for batch_id in range(1, 6)
}

for batch_id, batch_df in batches.items():
    print(f"Batch {batch_id}: {batch_df.count()} records")

Batch 1: 100 records
Batch 2: 100 records
Batch 3: 100 records
Batch 4: 100 records
Batch 5: 100 records


# 9. Control-Table Helper Functions

We use helper functions to record pipeline state.

The implementation is intentionally simple so the recovery logic remains visible.

In [0]:
def upsert_control(
    batch_id,
    status,
    attempt,
    records_processed=None,
    error_type=None,
    error_message=None
):
    now = datetime.utcnow()

    row = [(
        batch_id,
        status,
        attempt,
        now,
        now if status in ("SUCCESS", "FAILED") else None,
        records_processed,
        error_type,
        error_message
    )]

    incoming = spark.createDataFrame(row, control_schema)
    target = DeltaTable.forName(spark, CONTROL_TABLE)

    (
        target.alias("t")
        .merge(
            incoming.alias("s"),
            "t.batch_id = s.batch_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


def audit_event(batch_id, event_type, details):
    row = [(
        str(uuid.uuid4()),
        batch_id,
        datetime.utcnow(),
        event_type,
        details
    )]

    spark.createDataFrame(row, audit_schema)         .write.format("delta")         .mode("append")         .saveAsTable(AUDIT_TABLE)

# 10. Idempotent Batch Writer

This is the core recovery mechanism.

Instead of:

```text
INSERT every replayed record
```

we use:

```text
MERGE on order_id
```

Therefore replaying the same batch does not create duplicate business keys.

In [0]:
def write_batch_idempotently(batch_df):
    target = DeltaTable.forName(spark, ORDERS_TABLE)

    (
        target.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.order_id = s.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# 11. Process a Successful Batch

First demonstrate the normal path:

```text
PENDING
  ↓
RUNNING
  ↓
MERGE
  ↓
SUCCESS
```

In [0]:
def process_successfully(batch_id):
    batch_df = batches[batch_id]

    upsert_control(
        batch_id=batch_id,
        status="RUNNING",
        attempt=1
    )

    audit_event(
        batch_id,
        "BATCH_STARTED",
        "Normal processing started"
    )

    write_batch_idempotently(batch_df)

    upsert_control(
        batch_id=batch_id,
        status="SUCCESS",
        attempt=1,
        records_processed=batch_df.count()
    )

    audit_event(
        batch_id,
        "BATCH_SUCCESS",
        f"Processed {batch_df.count()} records"
    )


process_successfully(1)

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.i

In [0]:
display(
    spark.table(CONTROL_TABLE)
    .orderBy("batch_id")
)

assert (
    spark.table(CONTROL_TABLE)
    .filter((F.col("batch_id") == 1) & (F.col("status") == "SUCCESS"))
    .count()
    == 1
)

assert spark.table(ORDERS_TABLE).count() == 100

print("Batch 1 success validation passed.")

batch_id,status,attempt,started_at,completed_at,records_processed,error_type,error_message
1,SUCCESS,1,2026-08-24T08:53:12.703Z,2026-08-24T08:53:12.703Z,100,null,null


Batch 1 success validation passed.


# 12. Simulate a Transient Failure

Batch 2 will fail before its write.

This represents something like:

```text
temporary cluster/network/service failure
```

Because the write has not happened, replay is safe.

In [0]:
batch_id = 2
attempt = 1

upsert_control(
    batch_id=batch_id,
    status="RUNNING",
    attempt=attempt
)

audit_event(
    batch_id,
    "BATCH_STARTED",
    "Attempt 1 started"
)

try:
    raise RuntimeError("SIMULATED_TRANSIENT_FAILURE")

except Exception as e:
    upsert_control(
        batch_id=batch_id,
        status="FAILED",
        attempt=attempt,
        error_type="TRANSIENT",
        error_message=str(e)
    )

    audit_event(
        batch_id,
        "BATCH_FAILED",
        json.dumps({
            "error_type": "TRANSIENT",
            "message": str(e)
        })
    )

    print("Expected simulated failure:", str(e))

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.i

Expected simulated failure: SIMULATED_TRANSIENT_FAILURE


In [0]:
control_2 = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("batch_id") == 2)
    .collect()[0]
)

assert control_2["status"] == "FAILED"
assert control_2["error_type"] == "TRANSIENT"

assert spark.table(ORDERS_TABLE).count() == 100

print("Batch 2 failure-state validation passed.")

Batch 2 failure-state validation passed.


# 13. Retry the Failed Batch

Because the failure was classified as transient:

```text
FAILED
  ↓
RETRY
  ↓
MERGE
  ↓
SUCCESS
```

In [0]:
batch_id = 2
attempt = 2
batch_df = batches[batch_id]

upsert_control(
    batch_id=batch_id,
    status="RUNNING",
    attempt=attempt
)

audit_event(
    batch_id,
    "RETRY_STARTED",
    f"Retry attempt {attempt}"
)

write_batch_idempotently(batch_df)

upsert_control(
    batch_id=batch_id,
    status="SUCCESS",
    attempt=attempt,
    records_processed=batch_df.count()
)

audit_event(
    batch_id,
    "RETRY_SUCCESS",
    f"Retry attempt {attempt} succeeded"
)

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.i

In [0]:
assert spark.table(ORDERS_TABLE).count() == 200

control_2 = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("batch_id") == 2)
    .collect()[0]
)

assert control_2["status"] == "SUCCESS"
assert control_2["attempt"] == 2

print("Transient failure recovery passed.")

Transient failure recovery passed.


# 14. Demonstrate Idempotent Replay

Now replay Batch 2 again deliberately.

A non-idempotent append would create another 100 rows.

Our MERGE should not.

In [0]:
before_replay = spark.table(ORDERS_TABLE).count()

write_batch_idempotently(batches[2])

after_replay = spark.table(ORDERS_TABLE).count()

print("Before replay:", before_replay)
print("After replay :", after_replay)

assert before_replay == after_replay == 200

duplicate_keys = (
    spark.table(ORDERS_TABLE)
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_keys == 0

print("Idempotent replay validation passed.")

Before replay: 200
After replay : 200
Idempotent replay validation passed.


# 15. Deterministic Failure — Schema Mismatch

Now simulate a different failure.

The target expects:

```text
amount DOUBLE
```

The incoming source sends:

```text
amount STRING
```

This is not a transient infrastructure problem.

Retrying unchanged input will fail again.

Correct policy:

```text
Reject
 ↓
Record failure
 ↓
Fix source/mapping
 ↓
Replay
```

In [0]:
bad_schema = StructType([
    StructField("order_id", LongType(), False),
    StructField("batch_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("amount", StringType(), False),
    StructField("status", StringType(), False)
])

bad_batch = spark.createDataFrame(
    [
        (300001, 3, 10, "250.50", "COMPLETED")
    ],
    bad_schema
)

target_schema = spark.table(ORDERS_TABLE).schema

target_types = {
    f.name: f.dataType.simpleString()
    for f in target_schema.fields
}

incoming_types = {
    f.name: f.dataType.simpleString()
    for f in bad_batch.schema.fields
}

type_changes = [
    {
        "column": c,
        "expected": target_types[c],
        "incoming": incoming_types[c]
    }
    for c in sorted(set(target_types) & set(incoming_types))
    if target_types[c] != incoming_types[c]
]

print("Type changes:", type_changes)

assert any(x["column"] == "amount" for x in type_changes)

Type changes: [{'column': 'amount', 'expected': 'double', 'incoming': 'string'}]


# 16. Reject the Deterministic Failure

We record the failure as deterministic.

No retry is attempted because the input itself violates the target contract.

In [0]:
batch_id = 3

upsert_control(
    batch_id=batch_id,
    status="FAILED",
    attempt=1,
    error_type="DETERMINISTIC_SCHEMA",
    error_message="Incoming amount type is STRING; target expects DOUBLE"
)

audit_event(
    batch_id,
    "BATCH_REJECTED",
    json.dumps({
        "error_type": "DETERMINISTIC_SCHEMA",
        "type_changes": type_changes,
        "action": "STOP_AND_FIX"
    })
)

print("Batch 3 rejected as deterministic schema failure.")

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),


Batch 3 rejected as deterministic schema failure.


In [0]:
control_3 = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("batch_id") == 3)
    .collect()[0]
)

assert control_3["status"] == "FAILED"
assert control_3["error_type"] == "DETERMINISTIC_SCHEMA"

print("Deterministic failure validation passed.")

Deterministic failure validation passed.


# 17. Fix the Source and Replay Batch 3

The source contract is now corrected.

The repaired batch uses the proper `DOUBLE` type.

In [0]:
fixed_batch_3 = spark.createDataFrame(
    [
        (
            300001,
            3,
            10,
            250.50,
            "COMPLETED"
        )
    ],
    orders_schema
)

write_batch_idempotently(fixed_batch_3)

upsert_control(
    batch_id=3,
    status="SUCCESS",
    attempt=2,
    records_processed=fixed_batch_3.count()
)

audit_event(
    3,
    "REPLAY_SUCCESS",
    "Schema corrected and batch replayed successfully"
)

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),


In [0]:
assert (
    spark.table(CONTROL_TABLE)
    .filter((F.col("batch_id") == 3) & (F.col("status") == "SUCCESS"))
    .count()
    == 1
)

assert (
    spark.table(ORDERS_TABLE)
    .filter(F.col("order_id") == 300001)
    .count()
    == 1
)

print("Schema failure recovery validation passed.")

Schema failure recovery validation passed.


# 18. Data-Level Failure and Quarantine

Not every invalid record should fail the entire batch.

Example:

```text
amount = -500
```

if our business rule says order amounts must be non-negative.

We will separate:

```text
Valid records → Delta target
Invalid records → Quarantine
```

In [0]:
batch_4 = spark.createDataFrame(
    [
        (400001, 4, 11, 150.0, "COMPLETED"),
        (400002, 4, 12, -500.0, "COMPLETED"),
        (400003, 4, 13, 275.0, "COMPLETED")
    ],
    orders_schema
)

valid_4 = batch_4.filter(F.col("amount") >= 0)
invalid_4 = batch_4.filter(F.col("amount") < 0)

print("Valid:", valid_4.count())
print("Invalid:", invalid_4.count())

assert valid_4.count() == 2
assert invalid_4.count() == 1

Valid: 2
Invalid: 1


In [0]:
quarantine_rows = invalid_4.select(
    F.lit(4).cast("int").alias("batch_id"),
    F.col("order_id"),
    F.lit("Amount must be non-negative").alias("reason"),
    F.lit("DATA_QUALITY").alias("error_type"),
    F.current_timestamp().alias("created_at")
)

quarantine_rows.write     .format("delta")     .mode("append")     .saveAsTable(QUARANTINE_TABLE)

write_batch_idempotently(valid_4)

upsert_control(
    batch_id=4,
    status="SUCCESS",
    attempt=1,
    records_processed=valid_4.count()
)

audit_event(
    4,
    "PARTIAL_SUCCESS",
    json.dumps({
        "valid_records": valid_4.count(),
        "quarantined_records": invalid_4.count()
    })
)

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),


In [0]:
assert (
    spark.table(ORDERS_TABLE)
    .filter(F.col("batch_id") == 4)
    .count()
    == 2
)

assert (
    spark.table(QUARANTINE_TABLE)
    .filter(F.col("batch_id") == 4)
    .count()
    == 1
)

print("Quarantine validation passed.")

Quarantine validation passed.


# 19. Simulate a Write-Succeeds / State-Update-Fails Scenario

This is one of the most important recovery cases.

We will:

1. write Batch 5 successfully
2. deliberately fail before recording SUCCESS
3. leave control state as RUNNING
4. recover by replaying Batch 5 idempotently

This demonstrates why pipeline state alone cannot be treated as proof that data was or was not committed.

In [0]:
batch_id = 5
batch_5 = batches[5]

upsert_control(
    batch_id=batch_id,
    status="RUNNING",
    attempt=1
)

audit_event(
    batch_id,
    "BATCH_STARTED",
    "Attempt 1 started"
)

write_batch_idempotently(batch_5)

audit_event(
    batch_id,
    "DATA_WRITE_COMPLETED",
    "Delta write completed before simulated application failure"
)

try:
    raise RuntimeError("SIMULATED_POST_WRITE_APPLICATION_FAILURE")
except Exception as e:
    print("Expected simulated application failure:", str(e))

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),


Expected simulated application failure: SIMULATED_POST_WRITE_APPLICATION_FAILURE


In [0]:
count_after_write_failure = (
    spark.table(ORDERS_TABLE)
    .filter(F.col("batch_id") == 5)
    .count()
)

control_5 = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("batch_id") == 5)
    .collect()[0]
)

print("Batch 5 records in Delta:", count_after_write_failure)
print("Control status:", control_5["status"])

assert count_after_write_failure == 100
assert control_5["status"] == "RUNNING"

print("Post-write failure scenario reproduced.")

Batch 5 records in Delta: 100
Control status: RUNNING
Post-write failure scenario reproduced.


# 20. Recover the Ambiguous Batch

We cannot assume:

```text
RUNNING = no data written
```

Instead we inspect the target and use an idempotent replay.

Because the MERGE uses `order_id` as the key, replay is safe.

In [0]:
batch_id = 5
attempt = 2

audit_event(
    batch_id,
    "RECOVERY_STARTED",
    "Inspecting and replaying ambiguous batch idempotently"
)

write_batch_idempotently(batch_5)

upsert_control(
    batch_id=batch_id,
    status="SUCCESS",
    attempt=attempt,
    records_processed=batch_5.count()
)

audit_event(
    batch_id,
    "RECOVERY_SUCCESS",
    "Ambiguous batch safely recovered"
)

/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/home/spark-c56d2e08-9b4e-40d9-8435-88/.ipykernel/83/command-7492533675006394-3670009648:41: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow(),


In [0]:
batch_5_count = (
    spark.table(ORDERS_TABLE)
    .filter(F.col("batch_id") == 5)
    .count()
)

duplicate_keys = (
    spark.table(ORDERS_TABLE)
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

control_5 = (
    spark.table(CONTROL_TABLE)
    .filter(F.col("batch_id") == 5)
    .collect()[0]
)

assert batch_5_count == 100
assert duplicate_keys == 0
assert control_5["status"] == "SUCCESS"
assert control_5["attempt"] == 2

print("Ambiguous write recovery passed.")

Ambiguous write recovery passed.


# 21. Retry Policy

A simple policy:

```text
Maximum attempts = 3
```

Example:

```text
Attempt 1 → FAIL
Attempt 2 → FAIL
Attempt 3 → SUCCESS
```

But deterministic failures should not consume retries indefinitely.

```text
Schema mismatch
     ↓
STOP
     ↓
FIX
     ↓
REPLAY
```

In [0]:
MAX_RETRIES = 3

def retry_allowed(error_type, attempt):
    if error_type == "TRANSIENT" and attempt < MAX_RETRIES:
        return True

    return False

assert retry_allowed("TRANSIENT", 1) is True
assert retry_allowed("TRANSIENT", 2) is True
assert retry_allowed("TRANSIENT", 3) is False
assert retry_allowed("DETERMINISTIC_SCHEMA", 1) is False

print("Retry policy validation passed.")

Retry policy validation passed.


# 22. Inspect Pipeline Control State

In [0]:
display(
    spark.table(CONTROL_TABLE)
    .orderBy("batch_id")
)

batch_id,status,attempt,started_at,completed_at,records_processed,error_type,error_message
1,SUCCESS,1,2026-08-24T08:53:12.703Z,2026-08-24T08:53:12.703Z,100,null,null
2,SUCCESS,2,2026-08-24T08:53:48.678Z,2026-08-24T08:53:48.678Z,100,null,null
3,SUCCESS,2,2026-08-24T08:54:13.685Z,2026-08-24T08:54:13.685Z,1,null,null
4,SUCCESS,1,2026-08-24T08:54:28.260Z,2026-08-24T08:54:28.260Z,2,null,null
5,SUCCESS,2,2026-08-24T08:54:54.001Z,2026-08-24T08:54:54.001Z,100,null,null


# 23. Inspect Quarantined Records

In [0]:
display(
    spark.table(QUARANTINE_TABLE)
    .orderBy("batch_id", "order_id")
)

batch_id,order_id,reason,error_type,created_at
4,400002,Amount must be non-negative,DATA_QUALITY,2026-08-24T08:54:24.202Z


# 24. Inspect Recovery Audit

In [0]:
display(
    spark.table(AUDIT_TABLE)
    .orderBy("batch_id", "event_timestamp")
)

event_id,batch_id,event_timestamp,event_type,details
51b3f1f3-a79d-4cb2-b54a-83200837afd7,1,2026-08-24T08:53:07.212Z,BATCH_STARTED,Normal processing started
1b84b81d-e978-4926-a024-34bed09d38f3,1,2026-08-24T08:53:18.083Z,BATCH_SUCCESS,Processed 100 records
0d340ff5-d43e-4d79-8c3d-879d93f74542,2,2026-08-24T08:53:30.361Z,BATCH_STARTED,Attempt 1 started
2ba86f58-0602-4b9b-9ff2-964dba5b0580,2,2026-08-24T08:53:36.334Z,BATCH_FAILED,"{""error_type"": ""TRANSIENT"", ""message"": ""SIMULATED_TRANSIENT_FAILURE""}"
2892c2b8-83dc-48fd-a0f2-87a5994bfc84,2,2026-08-24T08:53:43.949Z,RETRY_STARTED,Retry attempt 2
95b83cb6-badb-4668-843e-24e03e93e24d,2,2026-08-24T08:53:53.116Z,RETRY_SUCCESS,Retry attempt 2 succeeded
6e2faaae-7355-4d3f-80f7-9e2c622e9fce,3,2026-08-24T08:54:07.337Z,BATCH_REJECTED,"{""error_type"": ""DETERMINISTIC_SCHEMA"", ""type_changes"": [{""column"": ""amount"", ""expected"": ""double"", ""incoming"": ""string""}], ""action"": ""STOP_AND_FIX""}"
b5b3a182-f1a3-4357-aa1c-19985c9c3d72,3,2026-08-24T08:54:18.172Z,REPLAY_SUCCESS,Schema corrected and batch replayed successfully
288fc60b-f4c0-4cf1-a304-029457fb08ce,4,2026-08-24T08:54:32.516Z,PARTIAL_SUCCESS,"{""valid_records"": 2, ""quarantined_records"": 1}"
86535e7b-3ae0-4647-abab-5475bc04a698,5,2026-08-24T08:54:38.974Z,BATCH_STARTED,Attempt 1 started


# 25. Inspect Delta Transaction History

Pipeline recovery should be understandable from both:

```text
Pipeline control/audit
        +
Delta transaction history
```

Inspect the Delta history for the target.

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}")
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
)

version,timestamp,operation,operationParameters
7,2026-08-24T08:54:54.000Z,MERGE,"Map(predicate -> [""(order_id#27025L = order_id#27045L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
6,2026-08-24T08:54:45.000Z,MERGE,"Map(predicate -> [""(order_id#26329L = order_id#26349L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
5,2026-08-24T08:54:28.000Z,MERGE,"Map(predicate -> [""(order_id#24944L = order_id#24769L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
4,2026-08-24T08:54:13.000Z,MERGE,"Map(predicate -> [""(order_id#23682L = order_id#23702L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
3,2026-08-24T08:54:01.000Z,MERGE,"Map(predicate -> [""(order_id#22741L = order_id#22615L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
2,2026-08-24T08:53:49.000Z,MERGE,"Map(predicate -> [""(order_id#21751L = order_id#21771L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
1,2026-08-24T08:53:13.000Z,MERGE,"Map(predicate -> [""(order_id#19002L = order_id#19022L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])"
0,2026-08-24T08:52:50.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)"


# 26. Final Data Validation

Expected valid data:

```text
Batch 1 → 100
Batch 2 → 100
Batch 3 → 1
Batch 4 → 2 valid
Batch 5 → 100
```

Total:

```text
303 valid target records
```

Batch 4 has one quarantined record.

This demonstrates that recovery can preserve correctness while isolating bad data.

In [0]:
final_count = spark.table(ORDERS_TABLE).count()
quarantine_count = spark.table(QUARANTINE_TABLE).count()

print("Final target rows:", final_count)
print("Quarantined rows :", quarantine_count)

assert final_count == 303
assert quarantine_count == 1

duplicate_keys = (
    spark.table(ORDERS_TABLE)
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_keys == 0

print("Final data correctness validation passed.")

Final target rows: 303
Quarantined rows : 1
Final data correctness validation passed.


# 27. Final Control-State Validation

Expected:

```text
Batch 1 → SUCCESS
Batch 2 → SUCCESS after retry
Batch 3 → SUCCESS after schema fix/replay
Batch 4 → SUCCESS with one quarantined record
Batch 5 → SUCCESS after ambiguous-write recovery
```

In [0]:
control = spark.table(CONTROL_TABLE)

expected = {
    1: ("SUCCESS", 1),
    2: ("SUCCESS", 2),
    3: ("SUCCESS", 2),
    4: ("SUCCESS", 1),
    5: ("SUCCESS", 2)
}

for batch_id, (expected_status, expected_attempt) in expected.items():
    row = control.filter(F.col("batch_id") == batch_id).collect()

    assert len(row) == 1
    assert row[0]["status"] == expected_status
    assert row[0]["attempt"] == expected_attempt

print("All control states are correct.")

All control states are correct.


# 28. Final Recovery Validation Suite

In [0]:
assert spark.table(ORDERS_TABLE).count() == 303

assert (
    spark.table(ORDERS_TABLE)
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
    == 0
)

assert spark.table(QUARANTINE_TABLE).count() == 1
assert spark.table(AUDIT_TABLE).count() >= 1
assert spark.sql(f"DESCRIBE HISTORY {ORDERS_TABLE}").count() >= 1

assert (
    spark.table(CONTROL_TABLE)
    .filter(F.col("status") != "SUCCESS")
    .count()
    == 0
)

print("==============================================")
print("ALL BLOG 9 FAILURE RECOVERY TESTS PASSED")
print("==============================================")

ALL BLOG 9 FAILURE RECOVERY TESTS PASSED


# 29. Failure Recovery Decision Model

```text
                   FAILURE
                      │
                      ▼
                 CLASSIFY
                      │
        ┌─────────────┼─────────────┐
        ↓             ↓             ↓
    TRANSIENT    DETERMINISTIC   DATA-LEVEL
        │             │             │
        ↓             ↓             ↓
      RETRY          STOP        QUARANTINE
        │             │
        ↓             ↓
     REPLAY          FIX
        │             │
        └──────┬──────┘
               ↓
           VALIDATE
               ↓
            SUCCESS
```

# 30. The Most Important Recovery Scenario

```text
Delta write succeeds
        ↓
Application crashes
        ↓
Control state is stale
        ↓
Recovery inspects state
        ↓
Idempotent MERGE replay
        ↓
No duplicate records
        ↓
Control state updated
        ↓
SUCCESS
```

This is why these concepts are connected:

```text
Delta Transactions
        +
Control State
        +
Idempotency
        =
Safe Recovery
```

# 31. Production Checklist

## Failure handling

- [x] Identify failure categories
- [x] Distinguish transient vs deterministic failures
- [x] Track batch state
- [x] Record attempts
- [x] Implement retry policy
- [x] Replay failed batches
- [x] Handle ambiguous post-write failures
- [x] Quarantine invalid records
- [x] Audit recovery events

## Data correctness

- [x] Idempotent MERGE
- [x] Duplicate-key validation
- [x] Final record-count validation
- [x] Schema failure handling
- [x] Data-quality failure handling
- [x] Delta history inspection

## Production principle

> **Never assume that a failed application means a failed data transaction. Inspect state and design replay to be safe.**

# 32. Final Mental Model

```text
                 DATA PIPELINE
                      │
                      ▼
                   PROCESS
                      │
             ┌────────┴────────┐
             │                 │
          SUCCESS            FAILURE
             │                 │
             ▼                 ▼
          COMMIT            CLASSIFY
                               │
                    ┌──────────┼──────────┐
                    ↓          ↓          ↓
                  RETRY       FIX      QUARANTINE
                    │          │
                    └────┬─────┘
                         ↓
                       REPLAY
                         ↓
                    IDEMPOTENT
                       WRITE
                         ↓
                     VALIDATE
                         ↓
                      SUCCESS

# 33. Blog 9 Complete

We have now covered:

```text
Blog 4 → Incremental Processing
Blog 5 → Idempotency
Blog 6 → Data Quality
Blog 7 → Schema Evolution
Blog 8 → Delta Maintenance
Blog 9 → Failure Recovery
```

The final step is:

# Blog 10 — End-to-End Data Engineering Project

That project will combine the major concepts into one production-style pipeline:

```text
Source
  ↓
Bronze
  ↓
Incremental Ingestion
  ↓
Schema Validation
  ↓
Data Quality
  ↓
Silver
  ↓
Idempotent Processing
  ↓
SCD / MERGE
  ↓
Gold
  ↓
Delta Maintenance
  ↓
Failure Recovery
  ↓
Monitoring / Audit
```